In [1]:
# testing part 1 with dummy data

import numpy as np
import pandas as pd
import anndata as ad
from mudata import MuData
from wnn_gpu import wnn_gpu

# Parameters
n_rna_cells = 10000
n_atac_cells = 9000
n_shared_cells = 8000
n_rna_genes = 2000
n_atac_peaks = 1000

# Generate shared cell names
shared_cells = [f"cell_{i:05d}" for i in range(n_shared_cells)]
rna_only_cells = [f"cell_rna_{i:05d}" for i in range(n_rna_cells - n_shared_cells)]
atac_only_cells = [f"cell_atac_{i:05d}" for i in range(n_atac_cells - n_shared_cells)]

# Full obs_names
rna_obs = shared_cells + rna_only_cells
atac_obs = shared_cells + atac_only_cells

# Create RNA modality AnnData
rna_data = ad.AnnData(
    X=np.random.rand(n_rna_cells, n_rna_genes).astype(np.float32),
    obs=pd.DataFrame(index=rna_obs),
    var=pd.DataFrame(index=[f"gene_{i}" for i in range(n_rna_genes)])
)

# Create ATAC modality AnnData
atac_data = ad.AnnData(
    X=np.random.rand(n_atac_cells, n_atac_peaks).astype(np.float32),
    obs=pd.DataFrame(index=atac_obs),
    var=pd.DataFrame(index=[f"peak_{i}" for i in range(n_atac_peaks)])
)

# Wrap into MuData
mdata = MuData({"rna": rna_data, "atac": atac_data})

# Run WNN alignment pipeline
mdata_aligned = wnn_gpu(mdata, seed=123)

# --- Print results ---
print("\nAlignment Report (Large Data):")
for k, v in mdata_aligned.uns["wnn_alignment"].items():
    print(f"{k}: {v}")

# Confirm that the obs_names match
aligned_rna_obs = mdata_aligned.mod["rna"].obs_names
aligned_atac_obs = mdata_aligned.mod["atac"].obs_names

print("\nFinal aligned cell count:", len(aligned_rna_obs))
print("Matching obs names?", aligned_rna_obs.equals(aligned_atac_obs))

# Check for GPU or CPU
try:
    import cupy as cp
    device = cp.cuda.runtime.getDevice()
    device_name = cp.cuda.runtime.getDeviceProperties(device)['name'].decode('utf-8')
    print(f"\n GPU is available. Using device: {device_name}")
except ImportError:
    print("\n CuPy not installed — falling back to CPU (NumPy).")
except Exception as e:
    print(f"\n GPU check failed, using CPU. Reason: {str(e)}")


/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)



Alignment Report (Large Data):
rna_n_before: 10000
atac_n_before: 9000
n_intersection: 8000
n_only_rna: 2000
n_only_atac: 1000
kept_fraction: 0.8
order_hash: 695b34e048a7e7fa388aa97f19e0ee7717b45ce8
notes: Cells present in both modalities were kept; others dropped.

Final aligned cell count: 8000
Matching obs names? True

 GPU is available. Using device: NVIDIA L40S


In [ ]:
# testing part 2 



In [19]:
# testing part 3

import muon as mu
import anndata
import numpy as np
import scipy.sparse as sp
import _fusion_portable as fusion  # your Person 3 code
import cupy as cp
import rmm

# -----------------------------
# 1) Create dummy MuData with connectivity matrices
# -----------------------------
n_cells = 10
n_features_rna = 5
n_features_atac = 6

adata_rna = anndata.AnnData(np.random.rand(n_cells, n_features_rna))
adata_atac = anndata.AnnData(np.random.rand(n_cells, n_features_atac))

# Dummy KNN connectivities: CSR identity matrices
adata_rna.obsp["connectivities"] = sp.eye(n_cells, dtype=np.float32, format="csr")
adata_atac.obsp["connectivities"] = sp.eye(n_cells, dtype=np.float32, format="csr")

mdata = mu.MuData({"rna": adata_rna, "atac": adata_atac})

print("Backend:", fusion.backend_name())

# -----------------------------
# 2) Run Person 3 fusion
# -----------------------------
fusion.fuse_from_mudata(
    mdata,
    rna_key="rna",
    atac_key="atac",
    weight_mode="uniform",   # test different modes if desired
    temperature=1.0
)

# -----------------------------
# 3) Inspect outputs
# -----------------------------
print("Fused adjacency shape:", mdata.obsp["wnn_connectivities"].shape)
print("RNA weights:", mdata.obs["wnn_weight_rna"])
print("ATAC weights:", mdata.obs["wnn_weight_atac"])


Backend: GPU (CuPy + cuSPARSE)


/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:963: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pu

TypeError: 'rmm.pylibrmm.memory_resource.CudaMemoryResource' object is not callable

In [2]:
# TESTING PART 1 + 3

import wnn_gpu as wnn_gpu
import _fusion_portable as fusion
from muon import MuData
from anndata import AnnData
import numpy as np
from scipy.sparse import csr_matrix

# Dummy data
n_cells = 50
n_genes_rna = 30
n_genes_atac = 20
obs_names = [f"C{i}" for i in range(n_cells)]

rna = AnnData(np.random.rand(n_cells, n_genes_rna).astype(np.float32), obs={'cell': obs_names})
rna.obsp['connectivities'] = csr_matrix(np.random.rand(n_cells, n_cells).astype(np.float32))

atac = AnnData(np.random.rand(n_cells, n_genes_atac).astype(np.float32), obs={'cell': obs_names})
atac.obsp['connectivities'] = csr_matrix(np.random.rand(n_cells, n_cells).astype(np.float32))

mdata = MuData({'rna': rna, 'atac': atac})


# Person 1 alignment ----
mdata_aligned = wnn_gpu.wnn_gpu(mdata, modalities=("rna", "atac"), seed=42)
print("Alignment report:", mdata_aligned.uns["wnn_alignment"])


# NEED TO SET ALLOCATOR TO NONE -- WITHOUT THIS WE GET A TYPE ERROR
import cupy as cp
cp.cuda.set_allocator(None)


# Person 3 WNN fusion ----
fusion.fuse_from_mudata(mdata_aligned)
print("Fused adjacency shape:", mdata_aligned.obsp["wnn_connectivities"].shape)
print("RNA weights:", mdata_aligned.obs["wnn_weight_rna"])
print("ATAC weights:", mdata_aligned.obs["wnn_weight_atac"])


Alignment report: {'rna_n_before': 50, 'atac_n_before': 50, 'n_intersection': 50, 'n_only_rna': 0, 'n_only_atac': 0, 'kept_fraction': 1.0, 'order_hash': 'cd56133ad980d64ac5f2338de0cc00dcac11ffbd', 'notes': 'Cells present in both modalities were kept; others dropped.'}
Fused adjacency shape: (50, 50)
RNA weights: 0     0.504224
1     0.493997
2     0.500355
3     0.499515
4     0.505740
5     0.498666
6     0.497650
7     0.501791
8     0.498479
9     0.503764
10    0.500150
11    0.499683
12    0.495039
13    0.498315
14    0.503245
15    0.500438
16    0.500707
17    0.501063
18    0.501456
19    0.493257
20    0.500895
21    0.501756
22    0.500949
23    0.494020
24    0.504829
25    0.502868
26    0.502401
27    0.501591
28    0.500047
29    0.496810
30    0.495132
31    0.500822
32    0.502785
33    0.500903
34    0.499812
35    0.501854
36    0.495047
37    0.499100
38    0.497355
39    0.498992
40    0.504507
41    0.495608
42    0.500586
43    0.508402
44    0.499382
45    0.496

/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:963: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pu

In [3]:
# testing skeleton pipeline 

import numpy as np
import scipy.sparse as sp
import muon as mu
from anndata import AnnData
import cupy as cp

# Import your combined WNN pipeline
from test_skeleton import wnn_gpu  # the integrated Person1+3 pipeline

# -------------------------------
# 1) Create a small synthetic MuData
# -------------------------------
n_cells = 50
n_genes_rna = 20
n_peaks_atac = 30
k = 5

# Dummy RNA data
X_rna = np.random.rand(n_cells, n_genes_rna).astype(np.float32)
adata_rna = AnnData(X=X_rna)
adata_rna.obsm["X_pca"] = np.random.rand(n_cells, 10).astype(np.float32)

# Dummy ATAC data
X_atac = np.random.rand(n_cells, n_peaks_atac).astype(np.float32)
adata_atac = AnnData(X=X_atac)
adata_atac.obsm["X_pca"] = np.random.rand(n_cells, 10).astype(np.float32)

# Create dummy connectivity matrices
adata_rna.obsp["connectivities"] = sp.csr_matrix(np.random.rand(n_cells, n_cells).astype(np.float32))
adata_atac.obsp["connectivities"] = sp.csr_matrix(np.random.rand(n_cells, n_cells).astype(np.float32))

# Build MuData
mdata = mu.MuData({"rna": adata_rna, "atac": adata_atac})

# -------------------------------
# 2) Run the combined WNN pipeline
# -------------------------------
mdata = wnn_gpu(
    mdata,
    modalities=("rna", "atac"),
    k=k,
    seed=42
)

# -------------------------------
# 3) Inspect results
# -------------------------------
print("WNN connectivities type:", type(mdata.obsp["wnn_connectivities"]))
print("WNN weights RNA (first 5):", mdata.obs["wnn_weight_rna"][:5])
print("WNN weights ATAC (first 5):", mdata.obs["wnn_weight_atac"][:5])
print("UMAP embedding shape:", mdata.obsm["X_wnn_umap"].shape)
print("Runtime info:", mdata.uns["wnn_runtime"])


/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:963: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pu

[2025-09-24 08:39:37.055] [CUML] [info] build_algo set to brute_force_knn because random_state is given
WNN connectivities type: <class 'scipy.sparse._csr.csr_matrix'>
WNN weights RNA (first 5): 0    0.5
1    0.5
2    0.5
3    0.5
4    0.5
Name: wnn_weight_rna, dtype: float32
WNN weights ATAC (first 5): 0    0.5
1    0.5
2    0.5
3    0.5
4    0.5
Name: wnn_weight_atac, dtype: float32
UMAP embedding shape: (50, 2)
Runtime info: {'total': 0.6759898662567139}


In [2]:
# test 2 and 3 together
# WORKING

import numpy as np
import anndata as ad
import muon as mu
from pathlib import Path

# Import Person 2 + Person 3
from main_knn_pipeline import run_person2_pipeline
from _fusion_portable import fuse_from_mudata, backend_name

# --------------------------
# Make dummy multimodal data
# --------------------------
n_cells = 300
n_genes = 50
n_peaks = 40

# Dummy RNA counts
rna = ad.AnnData(
    X=np.random.poisson(1.0, (n_cells, n_genes)).astype(np.float32),
    obs={"cell_id": [f"cell{i}" for i in range(n_cells)]},
    var={"gene": [f"gene{i}" for i in range(n_genes)]},
)

# Dummy ATAC counts
atac = ad.AnnData(
    X=np.random.poisson(1.0, (n_cells, n_peaks)).astype(np.float32),
    obs={"cell_id": [f"cell{i}" for i in range(n_cells)]},
    var={"peak": [f"peak{i}" for i in range(n_peaks)]},
)

mdata = mu.MuData({"rna": rna, "atac": atac})

# --------------------------
# Run Person 2 pipeline
# --------------------------
print("Running Person 2 pipeline…")
output_file = Path("person2_output.h5mu")
run_person2_pipeline(mdata, output_file=output_file, k=10, n_components=15)

# --------------------------
# Reload file
# --------------------------
print(f"\nReloading {output_file}")
mdata_loaded = mu.read_h5mu(output_file)

print("Keys in mdata_loaded.mod:", list(mdata_loaded.mod.keys()))
print("Keys in mdata_loaded.uns:", list(mdata_loaded.uns.keys()))

# --------------------------
# Run Person 3 fusion
# --------------------------
print("\nRunning Person 3 fusion…")
print("Backend:", backend_name())
fuse_from_mudata(mdata_loaded, weight_mode="entropy_inverse", temperature=1.0)

# --------------------------
# Inspect results
# --------------------------
print("\n--- Fusion Results ---")
print("mdata_loaded.obsp keys:", list(mdata_loaded.obsp.keys()))
print("WNN matrix shape:", mdata_loaded.obsp["wnn_connectivities"].shape)
print("First few RNA weights:", mdata_loaded.obs["wnn_weight_rna"].head())
print("First few ATAC weights:", mdata_loaded.obs["wnn_weight_atac"].head())

print("\nPerson 2 -> Person 3 pipeline ran successfully.")


Running Person 2 pipeline…

PART 2 PIPELINE: PCA + kNN ANALYSIS
Input: MuData object with n_obs × n_vars = 300 × 90
  2 modalities
    rna:	300 x 50
      obs:	'cell_id'
      var:	'gene'
    atac:	300 x 40
      obs:	'cell_id'
      var:	'peak'
Parameters: n_components=15, k=10

[PHASE 1] GPU Environment Setup
------------------------------
📦 Checking basic packages...
✅ Basic packages available
🔧 Fixing CUDA libraries...
✅ Loaded 0 libraries: 
⚠️ Failed to load 9 libraries: libcudart.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_runtime/lib/libcudart.so.12), libnvrtc.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12), libnvrtc-builtins.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc-builtins.so.12), libnvjitlink.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/nvjitlink/lib/libnvjitlink.so.12), libcublas.

/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:963: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  warnings.warn(
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pu

✅ Saved to: person2_output.h5mu

PART 2 PIPELINE COMPLETE
Handoff to PART 3:
- RNA embedding: [300, 15]
- ATAC embedding: [300, 15]
- RNA connectivity: [300, 300]
- ATAC connectivity: [300, 300]

Next: PART 3 will compute per-cell weights and fuse matrices into WNN graph.

Reloading person2_output.h5mu
Keys in mdata_loaded.mod: ['rna', 'atac']
Keys in mdata_loaded.uns: ['affinity_summary', 'knn_summary', 'pca_summary', 'person2_handoff', 'qc_results']

Running Person 3 fusion…
Backend: GPU (CuPy + cuSPARSE)

--- Fusion Results ---
mdata_loaded.obsp keys: ['wnn_connectivities']
WNN matrix shape: (300, 300)
First few RNA weights: 0    0.445546
1    0.351179
2    0.720666
3    0.598655
4    0.418699
Name: wnn_weight_rna, dtype: float32
First few ATAC weights: 0    0.554454
1    0.648821
2    0.279334
3    0.401345
4    0.581301
Name: wnn_weight_atac, dtype: float32

Person 2 -> Person 3 pipeline ran successfully.


In [16]:
# testing 1 -> 2 -> 3
## WORKING

import numpy as np
import pandas as pd
import muon as mu
from mudata import MuData
from anndata import AnnData
from pathlib import Path

from wnn_gpu import wnn_gpu  # Part 1
from main_knn_pipeline import run_person2_pipeline  # Part 2
from _fusion_portable import fuse_from_mudata, backend_name  # Part 3

# -----------------------------
# Create a small dummy MuData
# -----------------------------
n_cells = 50
n_rna_features = 100
n_atac_features = 120

# RNA modality
X_rna = np.random.rand(n_cells, n_rna_features).astype(np.float32)
obs_rna = pd.DataFrame(index=[f"cell{i}" for i in range(n_cells)])
var_rna = pd.DataFrame(index=[f"gene{i}" for i in range(n_rna_features)])
adata_rna = AnnData(X=X_rna, obs=obs_rna, var=var_rna)

# ATAC modality
X_atac = np.random.rand(n_cells, n_atac_features).astype(np.float32)
obs_atac = pd.DataFrame(index=[f"cell{i}" for i in range(n_cells)])
var_atac = pd.DataFrame(index=[f"peak{i}" for i in range(n_atac_features)])
adata_atac = AnnData(X=X_atac, obs=obs_atac, var=var_atac)

# Combine into MuData
mdata = MuData({"rna": adata_rna, "atac": adata_atac})
print(f"Dummy MuData created: {mdata.n_obs} cells, RNA={mdata['rna'].n_vars}, ATAC={mdata['atac'].n_vars}")

# -----------------------------
# Part 1: Validate & align modalities
# -----------------------------
print("\nPart 1: Running wnn_gpu (Person 1)")
mdata = wnn_gpu(mdata, seed=42)

print("Part 1 complete!")

# -----------------------------
# Part 2: PCA + KNN Graphs
# -----------------------------
print("\nPart 2: Running Person 2 pipeline")
output_file_part2 = Path("dummy_person2_output.h5mu")
run_person2_pipeline(
    mdata,
    output_file=output_file_part2,
    k=5,          # smaller k for dummy
    n_components=10
)

print("Part 2 complete!")

# --------------------------
# Run Person 3 fusion
# --------------------------
print("\nRunning Person 3 fusion…")
print("Backend:", backend_name())
fuse_from_mudata(mdata_part2, weight_mode="entropy_inverse", temperature=1.0)

# --------------------------
# Inspect results
# --------------------------
print("\n--- Fusion Results ---")
print("mdata_loaded.obsp keys:", list(mdata_part2.obsp.keys()))
print("WNN matrix shape:", mdata_part2.obsp["wnn_connectivities"].shape)
print("First few RNA weights:", mdata_part2.obs["wnn_weight_rna"].head())
print("First few ATAC weights:", mdata_part2.obs["wnn_weight_atac"].head())


print("\nFull dummy pipeline test completed successfully!")


Dummy MuData created: 50 cells, RNA=100, ATAC=120

Part 1: Running wnn_gpu (Person 1)
Part 1 complete!

Part 2: Running Person 2 pipeline

PART 2 PIPELINE: PCA + kNN ANALYSIS
Input: MuData object with n_obs × n_vars = 50 × 220
  uns:	'wnn_params', 'wnn_alignment', 'wnn_runtime'
  2 modalities
    rna:	50 x 100
    atac:	50 x 120
Parameters: n_components=10, k=5

[PHASE 1] GPU Environment Setup
------------------------------
📦 Checking basic packages...
✅ Basic packages available
🔧 Fixing CUDA libraries...
✅ Loaded 0 libraries: 
⚠️ Failed to load 9 libraries: libcudart.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_runtime/lib/libcudart.so.12), libnvrtc.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12), libnvrtc-builtins.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc-builtins.so.12), libnvjitlink.so.12 (not found at /home/shadeform/.loc

/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/m

In [6]:
import time
import numpy as np
import pandas as pd
import muon as mu
from mudata import MuData
from anndata import AnnData
from pathlib import Path

from wnn_gpu import wnn_gpu  # Part 1
from main_knn_pipeline import run_person2_pipeline  # Part 2
from _fusion_portable import fuse_from_mudata, backend_name  # Part 3

def run_dummy_pipeline_large(n_cells=8000, n_rna_features=1000, n_atac_features=1200, k=30, n_components=50):
    timings = {}

    # -----------------------------
    # Create large dummy MuData
    # -----------------------------
    t0 = time.perf_counter()
    X_rna = np.random.rand(n_cells, n_rna_features).astype(np.float32)
    obs_rna = pd.DataFrame(index=[f"cell{i}" for i in range(n_cells)])
    var_rna = pd.DataFrame(index=[f"gene{i}" for i in range(n_rna_features)])
    adata_rna = AnnData(X=X_rna, obs=obs_rna, var=var_rna)

    X_atac = np.random.rand(n_cells, n_atac_features).astype(np.float32)
    obs_atac = pd.DataFrame(index=[f"cell{i}" for i in range(n_cells)])
    var_atac = pd.DataFrame(index=[f"peak{i}" for i in range(n_atac_features)])
    adata_atac = AnnData(X=X_atac, obs=obs_atac, var=var_atac)

    mdata = MuData({"rna": adata_rna, "atac": adata_atac})
    timings["dummy_data"] = time.perf_counter() - t0
    print(f"Dummy MuData created: {mdata.n_obs} cells, RNA={mdata['rna'].n_vars}, ATAC={mdata['atac'].n_vars}")

    # -----------------------------
    # Part 1: wnn_gpu
    # -----------------------------
    t0 = time.perf_counter()
    mdata = wnn_gpu(mdata, seed=42)
    timings["part1"] = time.perf_counter() - t0
    print("Part 1 complete!")

    # -----------------------------
    # Part 2: PCA + KNN Graphs
    # -----------------------------
    t0 = time.perf_counter()
    output_file_part2 = Path("dummy_person2_output_large.h5mu")
    mdata_part2 = run_person2_pipeline(
        mdata,
        output_file=output_file_part2,
        k=k,
        n_components=n_components
    )
    timings["part2"] = time.perf_counter() - t0
    print("Part 2 complete!")

    # -----------------------------
    # Part 3: WNN Fusion
    # -----------------------------
    t0 = time.perf_counter()
    print("Running Part 3 fusion… Backend:", backend_name())
    fuse_from_mudata(mdata_part2, weight_mode="entropy_inverse", temperature=1.0)
    timings["part3"] = time.perf_counter() - t0

    # -----------------------------
    # Inspect results
    # -----------------------------
    print("\n--- Fusion Results ---")
    print("mdata.obsp keys:", list(mdata_part2.obsp.keys()))
    if "wnn_connectivities" in mdata_part2.obsp:
        print("WNN matrix shape:", mdata_part2.obsp["wnn_connectivities"].shape)
    if "wnn_weight_rna" in mdata_part2.obs:
        print("First few RNA weights:\n", mdata_part2.obs["wnn_weight_rna"].head())
    if "wnn_weight_atac" in mdata_part2.obs:
        print("First few ATAC weights:\n", mdata_part2.obs["wnn_weight_atac"].head())

    print("\nFull large dummy pipeline test completed successfully!")
    return mdata_part2, timings

# -----------------------------
# Run the large dummy pipeline
# -----------------------------
mdata_final, timings = run_dummy_pipeline_large()
print("\n--- Step timings (seconds) ---")
for step, t in timings.items():
    print(f"{step}: {t:.4f}")


/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Dummy MuData created: 8000 cells, RNA=1000, ATAC=1200
Part 1 complete!

PART 2 PIPELINE: PCA + kNN ANALYSIS
Input: MuData object with n_obs × n_vars = 8000 × 2200
  uns:	'wnn_params', 'wnn_alignment', 'wnn_runtime'
  2 modalities
    rna:	8000 x 1000
    atac:	8000 x 1200
Parameters: n_components=50, k=30

[PHASE 1] GPU Environment Setup
------------------------------
📦 Checking basic packages...
✅ Basic packages available
🔧 Fixing CUDA libraries...
✅ Loaded 0 libraries: 
⚠️ Failed to load 9 libraries: libcudart.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_runtime/lib/libcudart.so.12), libnvrtc.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12), libnvrtc-builtins.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/cuda_nvrtc/lib/libnvrtc-builtins.so.12), libnvjitlink.so.12 (not found at /home/shadeform/.local/lib/python3.10/site-packages/nvidia/nvjitlink/lib/libn

/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/shadeform/miniconda3/envs/rapids-25.08/lib/python3.13/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)
